# Notebook 1 · Let PyTorch do the maths

Part of the hands-on session **Deep Learning, Hands-on**. The page with the activities: **https://gradlabs.in/learn/deep-learning/**

You will:
1. meet **tensors**, PyTorch's lists of numbers,
2. build the neuron from the page: `output = weight × input + bias`,
3. work out the **loss**, and let PyTorch find the **gradients**,
4. walk downhill with **gradient descent**, first by hand, then the way real networks do it.

**How to use a notebook:** click a cell, then press **Shift + Enter** to run it. Run the cells from top to bottom.

## 0 · Check for a GPU

In [ ]:
import torch
print("PyTorch", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print("✓ GPU found:", torch.cuda.get_device_name(0))
else:
    print("No GPU. It still works, just slower.")
    print("To turn it on: Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

## 1 · Tensors

A **tensor** is a list (or a grid) of numbers that PyTorch can do maths on. Maths on a tensor happens to every number at once.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print(x)
print(x * 2)        # every number doubled
print(x.mean())     # the average
print(x.shape)      # how many numbers: 5

## 2 · The neuron

The same dots as on the page: hours studied → test score.

In [ ]:
hours  = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
scores = torch.tensor([3.0, 5.0, 7.0, 9.0, 11.0])

weight, bias = 0.5, 1.0
guesses = weight * hours + bias
print("guesses:", guesses)
print("real:   ", scores)

## 3 · The loss

`loss = average of (guess − real)²`. ✏️ Change `weight` and `bias` above, run both cells again, and get the loss to 0.

In [ ]:
loss = ((guesses - scores) ** 2).mean()
print("loss:", round(loss.item(), 2))

## 4 · Let PyTorch find the gradient

`requires_grad=True` tells PyTorch to track every sum done with this number. Then `loss.backward()` works out the gradient: which way, and how steeply, the loss goes up when this number goes up.

Starting at weight 0 and bias 0, you should see −50 and −14, the same as on the page.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

loss = ((w * hours + b - scores) ** 2).mean()
loss.backward()

print("loss:", round(loss.item(), 2))
print("gradient for weight:", round(w.grad.item(), 2))
print("gradient for bias:  ", round(b.grad.item(), 2))

Both gradients are negative: raising the weight or the bias makes the loss go **down**. So we step up.

## 5 · Walk downhill, by hand

`new weight = weight − learning rate × gradient`, repeated. Two details:
- `torch.no_grad()` means "don't track this step": it's the update, not part of the loss.
- `.grad.zero_()` clears the old gradient, because PyTorch adds new gradients on top of old ones.

In [ ]:
learning_rate = 0.05
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

for step in range(1, 101):
    loss = ((w * hours + b - scores) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    w.grad.zero_()
    b.grad.zero_()
    if step in (1, 2, 5, 10, 50, 100):
        print(f"step {step:3}: weight {w.item():.3f}, bias {b.item():.3f}, loss {loss.item():.4f}")

✏️ **Try:** set `learning_rate = 0.1` and run it again. What happens to the loss? Then try `0.01`.

## 6 · The way real networks do it

Real code doesn't update each weight by hand. It uses two helpers:
- `nn.Linear(1, 1)`: a neuron, with its own weight and bias inside.
- an **optimizer**: does the "step downhill" for every weight at once. `SGD` is plain gradient descent, exactly what you did above.

The loop is the same five lines from the page.

In [ ]:
from torch import nn

model = nn.Linear(1, 1)                 # one neuron: weight × input + bias
loss_fn = nn.MSELoss()                  # average of (guess − real)²
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

X = hours.unsqueeze(1)                  # nn.Linear wants one row per example: shape (5, 1)
Y = scores.unsqueeze(1)

for step in range(500):
    guesses = model(X)                  # 1 · guess
    loss = loss_fn(guesses, Y)          # 2 · how wrong
    optimizer.zero_grad()               # 3 · clear old gradients
    loss.backward()                     # 4 · gradients
    optimizer.step()                    # 5 · step downhill

print("weight:", round(model.weight.item(), 3), " bias:", round(model.bias.item(), 3), " loss:", round(loss.item(), 5))

It should be close to weight 2 and bias 1. Here is the line it found:

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(hours, scores, label="real scores")
xs = torch.linspace(0, 6, 50).unsqueeze(1)
with torch.no_grad():
    plt.plot(xs, model(xs), color="green", label="the neuron's line")
plt.xlabel("hours studied"); plt.ylabel("score"); plt.legend(); plt.show()

## ✅ Done

You trained a neuron with the same loop every network uses. Next: **Notebook 2**, a real network that recognises clothes. Go back to the page for the link.